# Module 5: Pair Inventory Features with Next-Year Outcomes

本模块把本期库存特征与同一公司下一年度的经营结果连接起来。
教学说明用中文，代码、注释、列名和运行提示用英文；继续采用普通循环、分组、筛选和保存。
可接在原 Notebook 后面，也可单独运行。目录必须与 Module 4 一致；无需新安装工具，不发出网络请求。

输入：Module 4 的完整审核表、47 行库存特征表、metadata。
输出：保留全部配对检查的工作表、筛选后的试点配对表、处理规则说明。

研究问题：本期存货增长相对收入增长较高，是否与下一年度毛利率下降、经营现金流收入比下降有关？
这里先建立可检查的数据关系，不计算模型、显著性或投资结论；三家公司用于验证流程。

**时间边界必须明确：**我们用所选年报的 filed 日期作为信息可用日期的代理。
年报常在下一财年开始后提交，因此下一年度全年结果包含一段年报提交前已经发生的经营活动。
本模块是年度财务特征与下一财年结果的回顾性关联样本，不是已经认证的完整全年预测或交易回测样本。
缓存里最早可见的 filing 也不保证是最初业绩公告或最早公开披露；历史口径审核继续保留 pending。


## Cell 1: Set Up the Project

沿用相同目录。Path.cwd() 是当前 Jupyter 的工作目录，不一定是文件目录。
你前面的输出为 [project root]；继续使用同一工作目录即可。
本格只导入工具和确定路径，不下载数据。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
print("Project directory:", PROJECT_ROOT)


## Cell 2: Read the Module 4 Tables

annual 是完整年度审核表，目前 54 行；features 是本期库存特征表，目前 47 行。
它们用途不同：本期样本使用 features，查找下一年度使用完整 annual。

parse_dates 恢复日期对象；sort_values 保证各家公司按年度排序。
如果只在 47 行特征表中 shift(-1)，AEO 被排除的 2013 年可能被跳过，2012 年会被错误地连到 2014 年。
因此下一期查找必须保留完整年度时间顺序，之后才检查是否合格。


In [ ]:
annual = pd.read_csv(
    PROCESSED_DIR / "pilot_annual_audit.csv",
    parse_dates=["period_start", "period_end", "filed"],
)
features = pd.read_csv(
    PROCESSED_DIR / "pilot_inventory_features.csv",
    parse_dates=["period_start", "period_end", "filed", "previous_period_end"],
)
module4_metadata = json.loads(
    (PROCESSED_DIR / "pilot_module4_metadata.json").read_text(encoding="utf-8")
)
annual = annual.sort_values(["ticker", "period_end"]).reset_index(drop=True)
features = features.sort_values(["ticker", "period_end"]).reset_index(drop=True)
print("Annual lookup rows:", len(annual))
print("Current feature rows:", len(features))


## Cell 3: Attach the Next Annual Period

future 先复制公司的 ticker 和当前 period_end，作为合并键。
循环从完整年度表为每个当前年度取出下一行的日期、比率和审核状态。

.groupby("ticker") 按公司分开处理，避免把其他公司的记录当作下一年度。
.shift(-1) 取同一公司下一行；Module 4 的 shift(1) 取上一行。
这里先取下一行，下一格还要检查是否真的是连续下一年度。
"next_" + column 为这些信息加前缀，避免与本期列混淆。

.merge(..., how="left") 保留全部 47 行本期特征，找不到下一期时保留为空。
validate="one_to_one" 要求公司和期间键唯一，避免意外重复合并放大样本；报错时应查重复记录，不要直接去掉这个参数。
next_filed 和 next_accn 用于追溯下一期结果，不能作为本期预测模型的输入。


In [ ]:
future = annual[["ticker", "period_end"]].copy()
for column in [
    "period_start", "period_end", "period_days", "filed", "accn",
    "gross_margin", "ocf_margin", "row_usable", "known_scope_break",
]:
    future["next_" + column] = annual.groupby("ticker")[column].shift(-1)

paired = features.merge(
    future, on=["ticker", "period_end"], how="left", validate="one_to_one"
)
display(paired[[
    "ticker", "period_end", "next_period_start", "next_period_end", "next_known_scope_break",
]].head(8))


## Cell 4: Check Annual Continuity, Scope, and Filing Timing

consecutive_next_year 同时要求：两个年度终点相隔 350–380 天，下一年度从本年度结束后的第 1 天开始。
因此缺了一年或出现短过渡年度时，不能只因为它是“下一行”就匹配。

signal_available_date 使用本期所选年报的 filed 日期作为代理，并非声称是最早公开披露时间。
filed_time_ok 要求该日期不早于本期年度结束，且早于下一年度结束。
available_before_target_start 额外记录本期年报是否在下一财年开始前已经提交。
本期年报提前于下一年度结束，只能说明目标年度尚未结束，不能说明整个下一年度都尚未发生。

days_after_target_start 是年报提交日期减去目标年度起始日期的日历天差，正数表示目标财年已经开始。
本期特征已通过 Module 4 筛查；pair_usable 再要求下一期完整、连续、不跨越已知口径断点且提交时点符合上述规则。
.eq(True) / .eq(False) 明确比较状态；没有下一期时不会把缺失状态当成已通过审核。


In [ ]:
end_gap = (paired["next_period_end"] - paired["period_end"]).dt.days
start_gap = (paired["next_period_start"] - paired["period_end"]).dt.days
paired["consecutive_next_year"] = end_gap.between(350, 380) & start_gap.eq(1)
paired["signal_available_date"] = paired["filed"]
paired["filed_time_ok"] = (
    (paired["signal_available_date"] >= paired["period_end"])
    & (paired["signal_available_date"] < paired["next_period_end"])
)
paired["available_before_target_start"] = (
    paired["signal_available_date"] < paired["next_period_start"]
)
paired["days_after_target_start"] = (
    paired["signal_available_date"] - paired["next_period_start"]
).dt.days
paired["pair_usable"] = (
    paired["next_period_end"].notna() & paired["consecutive_next_year"]
    & paired["next_row_usable"].eq(True)
    & paired["next_known_scope_break"].eq(False) & paired["filed_time_ok"]
)
display(paired.loc[~paired["pair_usable"], [
    "ticker", "period_end", "next_period_end", "next_row_usable",
    "next_known_scope_break", "consecutive_next_year", "filed_time_ok",
]])


## Cell 5: Calculate the Next-Year Outcomes

目标值使用“下一年度比率 - 本年度比率”，不是直接使用 Module 4 中本年度相对上一年度的变化。

next_gross_margin_change = 下一年度毛利率 - 本年度毛利率。
next_ocf_margin_change = 下一年度经营现金流收入比 - 本年度经营现金流收入比。
两者为负分别表示毛利率下降、经营现金流收入比下降。
后一项不等于现金流绝对金额下降，收入同时变化时两者可能方向不同。

比率和差额继续保存为小数，例如 -0.03 表示下降 3 个百分点。
先把不合格配对的目标变化留空，再生成下降标签。astype("boolean") 是可缺失的布尔类型。
不合格配对标签使用 pd.NA，不能把“没有下一年数据”误记为 False（没有下降）。

inventory_outpaced_sales 表示存货增长率高于收入增长率。
收入和存货都下降时这个条件也可能成立，因此它不等于存货绝对增加或已经积压。
所有 next_ 结果列用于目标、追溯或配对审核，不能放入本期预测特征。


In [ ]:
paired["next_gross_margin_change"] = (
    paired["next_gross_margin"] - paired["gross_margin"]
)
paired["next_ocf_margin_change"] = paired["next_ocf_margin"] - paired["ocf_margin"]
outcome_columns = ["next_gross_margin_change", "next_ocf_margin_change"]
paired.loc[~paired["pair_usable"], outcome_columns] = float("nan")

for outcome, label in [
    ("next_gross_margin_change", "gross_margin_declined"),
    ("next_ocf_margin_change", "ocf_margin_declined"),
]:
    paired[label] = (paired[outcome] < 0).astype("boolean")
    paired.loc[~paired["pair_usable"], label] = pd.NA
paired["inventory_outpaced_sales"] = paired["inventory_sales_gap"] > 0


## Cell 6: Review Target-Period Length and Timing

Module 4 的 period_length_changed 标记本期与上一期的年度长度差异。
现在还要标记下一年度与本期的长度差异，所以用 | 合并两个提醒。
pair_period_length_changed 仅用于审核和敏感性分析；不在这里自动把收入按天数缩放。
这个标记含下一年度信息，本期预测模型中不能直接当作已知特征。

只在合格配对中汇总时间提示。你的缓存预计显示：合格配对 43 个，目标财年开始前已提交的本期年报 0 个。
年报提交时下一财年已经开始约 38–61 个日历天。
因此，当前适合研究年度关联，不能表述为在下一财年开始前预测了完整全年表现。
即使这条日期规则通过，仍不意味着已排除所有历史重列或首次披露时间问题。


In [ ]:
paired["pair_period_length_changed"] = (
    paired["period_length_changed"]
    | ((paired["next_period_days"] - paired["period_days"]).abs() > 3)
)
valid = paired["pair_usable"]
print("Usable next-year pairs:", int(valid.sum()))
print("Available before target year starts:", int(
    paired.loc[valid, "available_before_target_start"].sum()
))
display(paired.loc[valid, "days_after_target_start"].agg(["min", "max"]))
display(paired.loc[valid, [
    "ticker", "period_end", "signal_available_date", "next_period_start",
    "next_period_end", "days_after_target_start", "pair_period_length_changed",
]].tail(6))


## Cell 7: Build the Pilot Paired Sample

next_year_sample 只保留合格配对，paired 仍保留全部 47 行及未通过的判断结果。
你的缓存预计得到 43 行：AEO 13、ANF 15、URBN 15。

减少的 4 行：三家公司的最新年度没有下一年度结果；AEO 2012-01-28 的目标年度是 2013-02-02，存在已知业务口径变化，因此不用于变化目标。
Module 4 已排除了 AEO 2013-02-02 作为本期特征；本模块还要排除进入这个年度的目标比较。

research_columns 展示本期库存增长差与下一年目标。没有下一年度结果是正常情况，不要用前值或 0 填补。
本模块不按结果好坏筛选样本，也不解释为投资建议；历史缺失与业务口径变化仍可能影响样本选择。


In [ ]:
next_year_sample = paired.loc[paired["pair_usable"]].copy()
research_columns = [
    "ticker", "period_end", "inventory_sales_gap", "next_period_end",
    "next_gross_margin_change", "next_ocf_margin_change",
    "gross_margin_declined", "ocf_margin_declined",
]
print("Pilot paired rows:", len(next_year_sample))
display(next_year_sample.groupby("ticker").size().to_frame("Paired Rows"))
display(next_year_sample[research_columns].tail(8))


## Cell 8: Save the Paired Sample and Processing Rules

保存完整配对检查表、合格试点配对表与 metadata，不覆盖 Module 4 文件。
完整检查表包含不合格配对的下一年度原始比率，便于检查；这些行的研究目标变化和下降标签已经留空。
再次运行会覆盖 Module 5 的同名输出。

metadata 保留 Module 4 来源说明和本模块研究定义。accounting_scope_verified、point_in_time_verified、full_year_forecast_ready 均为 False。
这些标记准确表达已经完成配对，但历史口径和真正预测时间窗还需进一步处理。

CSV 不保存 pandas 的可缺失布尔类型；以后读回下降标签时，可在 pd.read_csv 中指定 dtype={"gross_margin_declined": "boolean", "ocf_margin_declined": "boolean"}。


In [ ]:
paired.to_csv(
    PROCESSED_DIR / "pilot_next_year_pairing_audit.csv",
    index=False, date_format="%Y-%m-%d",
)
next_year_sample.to_csv(
    PROCESSED_DIR / "pilot_next_year_pairs.csv",
    index=False, date_format="%Y-%m-%d",
)
metadata = {
    "processed_at_utc": datetime.now(timezone.utc).isoformat(),
    "module4_metadata": module4_metadata,
    "research_design": "Retrospective association of annual inventory features with next fiscal-year outcomes.",
    "unit": "Ratios and changes stored as decimals; dates have daily precision.",
    "current_feature_rows": len(paired),
    "usable_pair_rows": len(next_year_sample),
    "signal_date_rule": "Selected annual filing date used as a proxy; not certified first public disclosure.",
    "pair_rule": "Consecutive complete target year, no known target scope break, current filing before target year end.",
    "target_overlap": "The target fiscal year may already have started when the current annual report is filed.",
    "gross_margin_target": "Next gross margin minus current gross margin.",
    "cash_flow_target": "Next OCF-to-revenue ratio minus current OCF-to-revenue ratio.",
    "accounting_scope_verified": False,
    "point_in_time_verified": False,
    "full_year_forecast_ready": False,
}
(PROCESSED_DIR / "pilot_module5_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)
print("Saved: pilot_next_year_pairing_audit.csv")
print("Saved: pilot_next_year_pairs.csv")
print("Saved: pilot_module5_metadata.json")


## Completion and Next Step

完成后有可追溯的 43 行试点年度配对，用于验证下一期目标构建。
下一模块可以先做探索性图表和个案核对，观察库存增长差与后续结果的关系；正式统计推断应扩展公司数量并审核历史口径后再做。
43 个公司年度来自 3 家公司，不是 43 家独立公司，不能因此宣称有可靠预测能力或交易收益。
如果最终目标是严格预测，需要重设信息截止日、目标时间窗，并控制历史重列与样本选择问题。
